# Memoization, Tabulation, and Repeated Work


```{contents}
:local:
:depth: 2
```


```{index} Save Answers to Repeated Questions, Not Arbitrary Intermediate State
```



```{index} Compare Three Executions of the Same Dependency Rule
```

## Compare Three Executions of the Same Dependency Rule


A repeated subproblem is a question with the same complete state, not just a similarly named function call. With F(0)=0, F(1)=1 and F(n)=F(n-1)+F(n-2), the integer n fully identifies the question. Fix that definition before sharing results. Changing the base values creates a different sequence and invalidates an old cache.

For F(4), the naive call tree asks for F(3) and F(2). F(3) asks for F(2) and F(1), so F(2) is expanded twice. Termination is already guaranteed by decreasing n; the problem is repeated work.

```text
F(4)
  F(3)
    F(2)
      F(1)
      F(0)
    F(1)
  F(2)
    F(1)
    F(0)
```

There are nine invocations including base cases. Only four calls are simultaneously active on the deepest path. A cache can remove repeated expansion without turning the recursion into constant-depth execution.


In [1]:
using System;
using System.Collections.Generic;
var cache = new Dictionary<int, long>();
int calls = 0, hits = 0, computed = 0;
long Fib(int n)
{
    calls++;
    if (n <= 1) return n;
    if (cache.TryGetValue(n, out long saved))
    {
        hits++;
        Console.WriteLine($"Hit F({n})={saved}");
        return saved;
    }
    long value = checked(Fib(n - 1) + Fib(n - 2));
    cache[n] = value;
    computed++;
    Console.WriteLine($"Store F({n})={value}");
    return value;
}
long result = Fib(4);
Console.WriteLine($"F(4)={result}; calls={calls}; hits={hits}; computed={computed}");


Store F(2)=1
Store F(3)=2
Hit F(2)=1
Store F(4)=3
F(4)=3; calls=7; hits=1; computed=3


The cell intentionally calls only a fixed valid argument. The complete APIs below retain domain and resource limits. Base cases are returned directly rather than stored; therefore computed counts non-base cache misses, not every call. Cache F(2) only after both dependencies return, otherwise another call could read an incomplete result.

| Event during F(4) | Completed cache afterward | Significance |
| --- | --- | --- |
| Finish F(2) within F(3) | 2→1 | First non-base state completed |
| Finish F(3) | 2→1, 3→2 | Its dependencies were available |
| Request F(2) from F(4) | Unchanged | Cache hit avoids another expansion |
| Finish F(4) | 2→1, 3→2, 4→3 | Requested result is now complete |

For n≥2, there are n-1 distinct non-base states. Each is expanded once and makes two recursive calls, so total invocations are the initial call plus twice the number of expansions: 2n-1. For n=4 this is seven. Exact counts depend on this implementation's decision not to cache base cases; the linear growth claim does not depend on adopting this exact convention.


```{index} Fill the Same States Without Recursive Calls
```

## Fill the Same States Without Recursive Calls


Tabulation stores the same mathematical answers but selects an order explicitly. For increasing i, both i-1 and i-2 already exist. Initialization supplies F(0) and F(1); it is not an incidental convenience. The small cell below makes every completed table entry visible.


In [1]:
using System;
int n = 4;
long[] table = new long[n + 1];
table[0] = 0;
if (n >= 1) table[1] = 1;
for (int i = 2; i <= n; i++)
{
    table[i] = checked(table[i - 1] + table[i - 2]);
    Console.WriteLine($"F({i})={table[i]}");
}
Console.WriteLine($"Table: {string.Join(",", table)}");


F(2)=1
F(3)=2
F(4)=3
Table: 0,1,1,2,3


Guided task: append the row for i=5, naming the two source entries and the result. Then describe why filling from n down to two would read entries that are still only default zero. Change n to zero and check why the guard on initialization of entry one is necessary.

```{dropdown} Check the dependency order
F(5)=F(4)+F(3)=3+2=5. Descending fill reaches F(5) before its non-base predecessors have been computed, so default zeros are not valid answers. When n is zero, the array contains only entry zero; writing entry one would be outside it.
```

For transfer, keep the recurrence but choose F(0)=2 and F(1)=1. Derive the first six values before adapting both strategies, and explain why the old cache must not be reused. The chapter homework uses another recurrence; first identify its states, base cases, dependency order and requested output. Rolling storage is appropriate when only the final value is needed, while a full table supports inspection of all earlier entries.


```{index} Save Answers to Repeated Questions, Not Arbitrary Intermediate State
```

## Save Answers to Repeated Questions, Not Arbitrary Intermediate State



Fibonacci gives a small example where termination alone does not prevent repeated work. Computing F(5) asks for F(4) and F(3); computing F(4) asks for F(3) again. A cache can retain the completed answer for a numeric argument and reuse it when that same subproblem returns. The key is sufficient because the recurrence and base cases are fixed for this computation.

Memoization starts from the requested state and recursively solves missing dependencies. Tabulation chooses an order in which needed dependencies have already been solved. For Fibonacci, increasing argument order works because F(i) depends on smaller arguments. These are two execution strategies for a stated dependency structure, not two unrelated tricks for making code faster.

| i | F(i) | Dependencies already available |
| --- | --- | --- |
| 0 | 0 | Base case |
| 1 | 1 | Base case |
| 2 | 1 | F(1), F(0) |
| 3 | 2 | F(2), F(1) |
| 4 | 3 | F(3), F(2) |
| 5 | 5 | F(4), F(3) |

The table invariant is that every earlier stored entry has its correct specified value before the next entry is computed. Storing a result before all dependencies have been completed can cache a partial value as if it were final. Reusing one global cache across requests with different policies can likewise mix different problems under the same key.

Rolling storage keeps only the two predecessor values when only the final Fibonacci result is required. It cannot simultaneously promise to return all earlier table entries without retaining or recomputing them. This is a tradeoff between requested output and retained state. Memoization also does not eliminate recursive stack depth simply because repeated arithmetic has been removed.

Guided exercise: derive F(6) from the table, then sketch which repeated calls a naive recursion would make. Label cache misses and cache hits separately from active call frames. Finally, change the base values and explain why a cache produced under the previous definition cannot be reused without identifying which definition its keys belong to.

For related explanation, see the further reading below.[^development-reading]


```{index} Memoization and Tabulation
```

## Memoization and Tabulation



A state identifies one smaller problem. A transition relates it to other states. Define the state, base cases, legal inputs, dependency order, and returned result before choosing a cache or table. Counting problems such as Fibonacci need a decomposition; minimum-coin optimization additionally needs optimal substructure.

```{index} Repeated Work Is Visible
```

## Repeated Work Is Visible




```{index} Repeated Work Is Visible
```

F(0)=0, F(1)=1, and F(n)=F(n−1)+F(n−2) for n≥2. The argument strictly decreases, but the recursion tree repeats arguments. Count every invocation, including base cases; computed counts only non-base expansions. Compare the same n, not unrelated examples.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var naive=FibPlanning.Fibonacci(8,false);
var memo=FibPlanning.Fibonacci(8);
Console.WriteLine($"naive: value={naive.Value}, calls={naive.Calls}, computed={naive.Computed}");
Console.WriteLine($"memo: value={memo.Value}, calls={memo.Calls}, hits={memo.Hits}, computed={memo.Computed}");
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


The naive call recurrence is C(0)=C(1)=1 and C(n)=1+C(n−1)+C(n−2); C(n)=2F(n+1)−1. Calls are exponential in n while active stack depth is Θ(n). This teaching implementation caps naive n at 25 to keep browser runs short; all Fibonacci APIs otherwise accept 0..92, the range that fits signed long. Invalid n is rejected before recursion.

```{index} Memoization Stores Complete States
```

## Memoization Stores Complete States




```{index} Memoization Stores Complete States
```

The cache key n fully identifies F(n) because the recurrence and base cases are fixed. Every cached value is a completed correct result. Compute a missing state recursively, then store it. Local cache ownership avoids mixing unrelated requests or caller-supplied incorrect entries.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(int n in new[]{0,1,2,8,40})
{
 var result=FibPlanning.Fibonacci(n);
 Console.WriteLine($"n={n}, value={result.Value}, calls={result.Calls}, hits={result.Hits}, computed={result.Computed}");
}
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


For n≥2 there are n−1 distinct non-base states, 2n−1 calls including cache hits, and max(0,n−3) hits. Base results are not cached here. Expected constant-time dictionary lookup gives Θ(n) time, Θ(n) cache storage, and Θ(n) stack space. These are distinct costs. A fresh invocation starts a fresh cache; memoization does not make the recursive stack disappear. Induction proves correctness: bases are correct, recursive dependencies are smaller, and their sum is F(n).

```{index} Tabulation Follows Dependency Order
```

## Tabulation Follows Dependency Order




```{index} Tabulation Follows Dependency Order
```

The invariant before iteration i is that entries 0..i−1 equal their specified Fibonacci values. Writing table[i] from i−1 and i−2 preserves it. Initialize entry one only when it exists; zero must work.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(int n in new[]{0,1,10})
{
 var table=FibPlanning.FibonacciTable(n);
 Console.WriteLine($"n={n}: [{string.Join(",",table)}], rolling={FibPlanning.FibonacciRolling(n)}");
}
Console.WriteLine($"F(92)={FibPlanning.FibonacciRolling(92)}");
try{FibPlanning.FibonacciTable(93);}catch(ArgumentOutOfRangeException){Console.WriteLine("overflow domain rejected");}
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


A full table costs Θ(n) time and Θ(n) storage without recursion. Rolling state keeps only the two needed predecessors, giving Θ(n) time and Θ(1) auxiliary storage when only the final value is requested. It cannot provide every historical table entry. Checked addition protects arithmetic; validating 0..92 also avoids computing an unnecessary F(93) on the final rolling update.

```{rubric} Footnotes
```


[^development-reading]: Further reading: [OpenDSA, Dynamic Programming](https://opendsa.org/OpenDSA/Books/CS4104/html/DynamicProgramming.html). The worked business scenarios and guided reasoning here are original teaching examples.
